In [7]:
from googleapiclient.discovery import build
from tqdm import tqdm
import pandas as pd
import isodate
import os
from dotenv import load_dotenv

In [8]:
load_dotenv()

API_KEY = os.getenv("YOUTUBE_API_KEY")
YOUTUBE = build('youtube', 'v3', developerKey=API_KEY)

PUBLISHED_AFTER = '2024-01-01T00:00:00Z'
MIN_DURATION = 1800  # 30 mins
MAX_DURATION = 7200  # 120 mins

In [9]:
# --- Step 1: Search YouTube ---
def search_video_ids(query, max_results=200):
    video_ids = []
    next_page_token = None

    while len(video_ids) < max_results:
        request = YOUTUBE.search().list(
            q=query,
            part="id",
            type="video",
            maxResults=50,
            publishedAfter=PUBLISHED_AFTER,
            regionCode='US',
            videoDuration='long',  # Only videos > 20 min
            pageToken=next_page_token
        )
        response = request.execute()

        for item in response.get('items', []):
            video_ids.append(item['id']['videoId'])

        next_page_token = response.get('nextPageToken')
        if not next_page_token:
            break

    return video_ids[:max_results]


In [10]:
def fetch_metadata(video_ids):
    all_data = []
    valid_video_ids = set()
    duration_count = 0

    for i in range(0, len(video_ids), 50):
        batch = video_ids[i:i+50]
        request = YOUTUBE.videos().list(
            part="snippet,contentDetails,statistics",
            id=",".join(batch)
        )
        response = request.execute()

        returned_ids = {item['id'] for item in response.get('items', [])}
        valid_video_ids.update(returned_ids)

        for item in response.get('items', []):
            try:
                duration = isodate.parse_duration(item['contentDetails']['duration']).total_seconds()
                if MIN_DURATION <= duration <= MAX_DURATION:
                    snippet = item['snippet']
                    stats = item.get('statistics', {})

                    all_data.append({
                        "videoId": item['id'],
                        "videoUrl": f"https://www.youtube.com/watch?v={item['id']}",
                        "title": snippet.get('title'),
                        "description": snippet.get('description', ''),
                        "publishDate": snippet.get('publishedAt'),
                        "channelTitle": snippet.get('channelTitle'),
                        "channelId": snippet.get('channelId'),
                        "tags": snippet.get('tags', []),
                        "categoryId": snippet.get('categoryId', None),
                        "liveBroadcastContent": snippet.get('liveBroadcastContent', ''),
                        "defaultAudioLanguage": snippet.get('defaultAudioLanguage', ''),
                        "duration_sec": int(duration),
                        "viewCount": int(stats.get('viewCount', 0)),
                        "likeCount": int(stats.get('likeCount', 0)),
                        "commentCount": int(stats.get('commentCount', 0)),
                    })
                else:
                    duration_count += 1
            except Exception as e:
                continue

    print(f"Valid metadata fetched for {len(valid_video_ids)} of {len(video_ids)} requested IDs.")
    print(f"Invalid Duration Count: {duration_count}")
    return pd.DataFrame(all_data)


In [11]:
# --- Step 3: Run Query Loop ---
queries = ["ai podcast"]
unique_ids = set()

for q in tqdm(queries):
    ids = search_video_ids(q, max_results=50)
    unique_ids.update(ids)

print(f"Total unique video IDs: {len(unique_ids)}")

100%|██████████| 1/1 [00:00<00:00,  1.45it/s]

Total unique video IDs: 50


In [12]:
# --- Step 4: Fetch and Save ---
metadata_df = fetch_metadata(list(unique_ids))
metadata_df.head()

Valid metadata fetched for 50 of 50 requested IDs.
Invalid Duration Count: 28


,videoId,videoUrl,title,description,publishDate,channelTitle,channelId,tags,categoryId,liveBroadcastContent,defaultAudioLanguage,duration_sec,viewCount,likeCount,commentCount
0,07twxy9TpQo,https://www.youtube.com/watch?v=07twxy9TpQo,The $10M Multipreneur: How To Get RICH In The ...,Cop The Stay Delusional Merch Now: https://cal...,2025-03-31T12:00:01Z,The Calum Johnson Show,UCuvjQYKukKjVyhSVxQibgOw,[],22,none,,4802,622936,19240,878
1,vjVr4w9Pltw,https://www.youtube.com/watch?v=vjVr4w9Pltw,AI’s Message to Humanity – A Documentary by Ar...,What happens when five artificial intelligence...,2025-05-04T18:00:47Z,A Podcast Run by AI,UCkKAizO4QDDzwkYK-KKKICQ,[],22,none,,3661,117736,4264,1345
2,DB9mjd-65gw,https://www.youtube.com/watch?v=DB9mjd-65gw,"Sam Altman on AGI, GPT-5, and what’s next — th...","On the first episode of the OpenAI Podcast, Sa...",2025-06-18T14:59:20Z,OpenAI,UCXZCJLdBC09xxGZ6gcdrc6A,"[OpenAI, ChatGPT, SamAltman]",28,none,en,2424,359765,9663,1176
3,5MWT_doo68k,https://www.youtube.com/watch?v=5MWT_doo68k,"OpenAI’s Sam Altman Talks ChatGPT, AI Agents a...","The AI revolution is here to stay, says Sam Al...",2025-04-12T11:00:06Z,TED,UCAuUUnT6oDeKwE6v1NGQxug,"[TEDTalk, TEDTalks, TED Talk, TED Talks, TED, ...",28,none,en,2850,1636308,28048,3535
4,_jl64f-821o,https://www.youtube.com/watch?v=_jl64f-821o,Our AI Future Is WAY WORSE Than You Think | Yu...,"Yuval Noah Harari, renowned historian and auth...",2024-10-28T10:00:34Z,Rich Roll,UCpjlh0e319ksmoOD7bQFSiw,"[rich roll, rich roll podcast, self-improvemen...",22,none,en,5864,1025768,19458,3697


In [13]:
# metadata_df.to_csv("data/raw/metadata.csv", index=False)
# print(f"Saved {len(metadata_df)} entries to metadata.csv")

In [14]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled, NoTranscriptFound, VideoUnavailable
import pandas as pd
from pathlib import Path
import re

In [15]:
# Sample video ID (replace with any real ID from metadata.csv)
video_id = "t58X0md283Y"  # Replace with an actual videoId

# Set up output path
output_dir = Path("data/raw/transcripts")
output_dir.mkdir(parents=True, exist_ok=True)
output_file = output_dir / f"{video_id}.csv"

In [16]:
def clean_transcript_text(text):
    text = text.lower()                          # lowercase
    text = re.sub(r'\[.*?\]', '', text)          # remove [bracketed text]
    text = re.sub(r'https?://\\S+', '', text)    # remove URLs
    text = re.sub(r'[^a-z0-9.,!?\\s]', '', text)  # remove non-alphanumeric characters except punct
    text = re.sub(r'\\s+', ' ', text).strip()     # normalize whitespace
    return text

In [22]:
# Try fetching English transcript
try:
    transcript = YouTubeTranscriptApi.get_transcript(video_id, languages=['en'])
    df = pd.DataFrame(transcript)
    df['clean_text'] = df['text'].apply(clean_transcript_text)
    df = df[['start', 'duration', 'text', 'clean_text']]
    df.to_csv(output_file, index=False)
    print(f"Transcript saved to {output_file}")

except TranscriptsDisabled:
    print("Captions are disabled for this video.")
except NoTranscriptFound:
    print("No English transcript available for this video.")
except VideoUnavailable:
    print("Video is unavailable.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

Transcript saved to data/raw/transcripts/t58X0md283Y.csv
